# Prueba de integración del proyecto Sudoku

Este notebook valida el proceso completo utilizando los módulos finales de la carpeta `app`.

```text
imagen
→ YOLO detecta el tablero
→ OpenCV corrige la perspectiva
→ se generan 81 celdas
→ la CNN reconoce vacío o número 1–9
→ se valida el puzzle
→ CNN + MRV + backtracking lo resuelven
→ se comprueba la solución
```

El notebook debe ejecutarse desde la carpeta `app`.

## 1. Importar librerías y funciones

In [ ]:
import cv2
import matplotlib.pyplot as plt

from tensorflow import keras
from ultralytics import YOLO

from procesar_sudoku import procesar_sudoku
from clasificar_celdas import clasificar_celdas
from resolver_sudoku import (
    puzzle_parcial_valido,
    resolver_sudoku as resolver_con_backtracking,
    sudoku_valido
)

## 2. Definir las rutas

In [ ]:
ruta_yolo = "models/best.pt"

ruta_class = (
    "models/best_class_0_9.keras"
)

ruta_juego = (
    "models/best_juego.keras"
)

ruta_imagen = (
    "../modelo_yolo/img_proc/images/test/"
    "sudoku_168_jpg.rf.pfhTWZY1G9sbBr6PPDKz.jpg"
)

## 3. Cargar los modelos

In [ ]:
modelo_yolo = YOLO(
    ruta_yolo
)

modelo_class = keras.models.load_model(
    ruta_class
)

modelo_juego = keras.models.load_model(
    ruta_juego
)

print("Modelo YOLO cargado")

print(
    "Modelo class:",
    modelo_class.input_shape,
    "→",
    modelo_class.output_shape
)

print(
    "Modelo juego:",
    modelo_juego.input_shape,
    "→",
    modelo_juego.output_shape
)

## 4. Comprobar las arquitecturas

In [ ]:
if tuple(
    modelo_class.input_shape[1:]
) != (28, 28, 1):

    raise ValueError(
        "El modelo class no recibe imágenes 28 × 28 × 1."
    )

if modelo_class.output_shape[-1] != 10:

    raise ValueError(
        "El modelo class no tiene 10 clases."
    )

if tuple(
    modelo_juego.input_shape[1:]
) != (9, 9, 19):

    raise ValueError(
        "El modelo juego no recibe una entrada 9 × 9 × 19."
    )

if tuple(
    modelo_juego.output_shape[1:]
) != (9, 9, 9):

    raise ValueError(
        "La salida del modelo juego no es la esperada."
    )

print("Arquitecturas correctas")

## 5. Cargar la imagen

In [ ]:
imagen = cv2.imread(
    ruta_imagen
)

if imagen is None:

    raise ValueError(
        "No se ha podido cargar la imagen."
    )

plt.figure(
    figsize=(7, 7)
)

plt.imshow(
    cv2.cvtColor(
        imagen,
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    "Imagen original"
)

plt.axis("off")
plt.show()

## 6. Detectar y procesar el tablero

`procesar_sudoku.py` utiliza YOLO para detectar el tablero y OpenCV para corregir su perspectiva y dividirlo en 81 celdas.

In [ ]:
tablero, celdas = procesar_sudoku(
    imagen,
    modelo_yolo
)

print(
    "Celdas generadas:",
    len(celdas)
)

plt.figure(
    figsize=(7, 7)
)

plt.imshow(
    cv2.cvtColor(
        tablero,
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    "Tablero corregido"
)

plt.axis("off")
plt.show()

## 7. Clasificar las 81 celdas

`clasificar_celdas.py` prepara las celdas y utiliza `best_class_0_9.keras`.

```text
0 = celda vacía
1–9 = número reconocido
```

In [ ]:
(
    puzzle,
    confianzas,
    contenido_detectado,
    celdas_preparadas
) = clasificar_celdas(
    celdas,
    modelo_class
)

print("Puzzle reconocido:")
print(puzzle)

print(
    "\nCeldas con número:",
    int(
        contenido_detectado.sum()
    )
)

print(
    "Celdas vacías:",
    int(
        81
        - contenido_detectado.sum()
    )
)

print(
    "Confianza media:",
    f"{confianzas.mean():.2%}"
)

In [ ]:
fig, axes = plt.subplots(
    9,
    9,
    figsize=(11, 11)
)

for indice, celda in enumerate(
    celdas_preparadas
):

    fila = indice // 9
    columna = indice % 9

    numero = puzzle[
        fila,
        columna
    ]

    confianza = confianzas[
        fila,
        columna
    ]

    axes[
        fila,
        columna
    ].imshow(
        celda,
        cmap="gray"
    )

    axes[
        fila,
        columna
    ].set_title(
        f"{numero} | {confianza:.0%}",
        fontsize=7
    )

    axes[
        fila,
        columna
    ].axis("off")

plt.tight_layout()
plt.show()

## 8. Validar el puzzle reconocido

In [ ]:
if not puzzle_parcial_valido(
    puzzle
):

    raise ValueError(
        "El puzzle reconocido contiene contradicciones."
    )

print(
    "Puzzle inicial válido"
)

## 9. Resolver el Sudoku

Aquí se ejecuta el **backtracking**.

La función `resolver_sudoku()` de `resolver_sudoku.py`:

1. utiliza MRV para elegir la celda con menos candidatos;
2. utiliza la CNN para ordenar los candidatos;
3. prueba el candidato más probable;
4. si aparece una contradicción, borra el número;
5. retrocede y prueba el siguiente candidato.

In [ ]:
solucion, estadisticas = (
    resolver_con_backtracking(
        puzzle,
        modelo_juego,
        max_nodos=100000
    )
)

if solucion is None:

    raise ValueError(
        "No se ha encontrado una solución."
    )

print("Sudoku resuelto:")
print(solucion)

print(
    "\nNodos:",
    estadisticas["nodos"]
)

print(
    "Retrocesos:",
    estadisticas["retrocesos"]
)

## 10. Validar y mostrar la solución

In [ ]:
valido = sudoku_valido(
    solucion
)

print(
    "Sudoku válido:",
    valido
)

if not valido:

    raise ValueError(
        "La solución obtenida no es válida."
    )

fig, ax = plt.subplots(
    figsize=(6, 6)
)

ax.set_xlim(0, 9)
ax.set_ylim(0, 9)
ax.invert_yaxis()

for i in range(10):

    grosor = (
        2.5
        if i % 3 == 0
        else 0.8
    )

    ax.plot(
        [0, 9],
        [i, i],
        linewidth=grosor
    )

    ax.plot(
        [i, i],
        [0, 9],
        linewidth=grosor
    )

for fila in range(9):
    for columna in range(9):

        ax.text(
            columna + 0.5,
            fila + 0.5,
            str(
                solucion[
                    fila,
                    columna
                ]
            ),
            ha="center",
            va="center",
            fontsize=18
        )

ax.set_title(
    "Sudoku resuelto"
)

ax.axis("off")
plt.show()

# Resultado de la prueba

La integración queda validada cuando:

- los tres modelos cargan correctamente;
- YOLO detecta el tablero;
- OpenCV genera 81 celdas;
- el clasificador genera un puzzle válido;
- el backtracking encuentra una solución;
- la solución cumple todas las reglas del Sudoku.